In [1]:
#!/usr/bin/env python3
"""
run_analysis.py — FLARE Single Model Full Analysis (v2.2)
==========================================================
Tek FLARE modeli + LR + RF/XGBoost.
Aynı FLARE modeli her analizde kullanılır.
Tüm hesaplama helpers.py'de.

KULLANIM:
  1. Aşağıdaki parametreleri değiştir
  2. python run_analysis.py
"""

import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from datasets import load_dataset
from flare import FLARE

from helpers import (
    # Eğitim
    fit_competitor_lr,
    fit_competitor_rf,
    compute_metrics,
    # Model özetleri
    print_model_summary,
    lr_coefficients_full,
    lr_vs_flare_table,
    beta_equiv_summary,
    # Öznitelik önem
    raw_coefficients_comparison,
    feature_importance_ranking,
    # Etkileşimler
    run_interaction_analysis,
    beta_equiv_by_group,
    # Hasta profili
    patient_risk_profile,
    print_patient_risk_report,
    rff_attribution_patient,
    # Hasta Hessian
    compute_patient_hessians_with_ci,
    print_patient_hessians_with_ci,
    global_vs_patient_interactions,
    unique_patient_interactions,
    # Kalibrasyon & validasyon
    residuals_summary,
    binary_posterior_coverage,
    patient_posterior_plausibility,
    print_coverage_report,
    # Görselleştirme verileri
    calibration_plot_data,
    residuals_plot_data,
    roc_curve_data,
    pr_curve_data,
    # Synthetic
    true_beta_comparison,
)

EPS = 1e-10

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False


# ═══════════════════════════════════════════════════════════════
#  KULLANICI PARAMETRELERİ
# ═══════════════════════════════════════════════════════════════
DATASET   = 'linear'
M         = 126
L2        = 0.1
SIGMA     = None
SEED      = 42
TEST_SIZE = 0.20


# ═══════════════════════════════════════════════════════════════
#  ANA ANALİZ
# ═══════════════════════════════════════════════════════════════

def main():

    # ──────────────────────────────────────────────────────────
    # 1. VERİ YÜKLE + SPLIT
    # ──────────────────────────────────────────────────────────
    X, y, meta = load_dataset(DATASET, normalize=True, seed=SEED)
    feature_names = meta.get(
        'feature_names',
        [f'x{i+1}' for i in range(X.shape[1])])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE,
        stratify=y, random_state=SEED)

    d = X_train.shape[1]

    print(f"\n  {'═' * 75}")
    print(f"  FLARE — TEK MODEL ANALİZ")
    print(f"  {'═' * 75}")
    print(f"  Veri       : {meta['name']}")
    print(f"  n_total    : {meta['n_final']}")
    print(f"  d          : {meta['d_final']}")
    print(f"  pos_rate   : {meta['pos_rate']:.1%}")
    print(f"  n_train    : {len(y_train)}")
    print(f"  n_test     : {len(y_test)}")
    print(f"  m          : {M}")
    print(f"  λ (l2)     : {L2}")
    print(f"  σ          : "
          f"{'otomatik' if SIGMA is None else f'{SIGMA:.4f}'}")
    print(f"  seed       : {SEED}")

    # ──────────────────────────────────────────────────────────
    # 2. FLARE EĞİT
    # ──────────────────────────────────────────────────────────
    print(f"\n  {'─' * 75}")
    print(f"  FLARE Eğitimi")
    print(f"  {'─' * 75}")

    flare = FLARE(m=M, l2=L2, seed=SEED)
    flare.fit(X_train, y_train, sigma=SIGMA)

    flare_probs = flare.predict_proba(X_test)[:, 1]
    flare_preds = flare.predict(X_test, threshold=0.5)
    flare_m = compute_metrics(y_test, flare_probs, flare_preds)

    print(f"  σ (seçilen): {flare.sigma_:.6f}")
    print(f"  AUC        : {flare_m['AUC']:.4f}")
    print(f"  PR-AUC     : {flare_m['PR-AUC']:.4f}")
    print(f"  F1         : {flare_m['F1']:.4f}")
    print(f"  Accuracy   : {flare_m['ACC']:.4f}")
    print(f"  MCC        : {flare_m['MCC']:.4f}")

    # ──────────────────────────────────────────────────────────
    # 3. LR EĞİT
    # ──────────────────────────────────────────────────────────
    print(f"\n  {'─' * 75}")
    print(f"  LR Eğitimi (Wald karşılaştırma için)")
    print(f"  {'─' * 75}")

    lr_model = fit_competitor_lr(X_train, y_train, C=1.0)
    lr_probs = lr_model.predict_proba(X_test)[:, 1]
    lr_m = compute_metrics(y_test, lr_probs,
                           lr_model.predict(X_test))
    print(f"  LR AUC     : {lr_m['AUC']:.4f}")

    # ──────────────────────────────────────────────────────────
    # 4. MODEL ÖZETİ
    # ──────────────────────────────────────────────────────────
    summary = print_model_summary(flare, X_test, y_test)

    # ──────────────────────────────────────────────────────────
    # 5. β_equiv TABLOSU
    # ──────────────────────────────────────────────────────────
    beq = beta_equiv_summary(flare, feature_names)

    # ──────────────────────────────────────────────────────────
    # 6. LR KATSAYILARI + KARŞILAŞTIRMA
    # ──────────────────────────────────────────────────────────
    lr_res = lr_coefficients_full(
        lr_model, X_train, y_train, feature_names)
    lr_vs_flare_table(lr_res, beq, feature_names)

    # ──────────────────────────────────────────────────────────
    # 7. ÖZNİTELİK ÖNEMİ: FLARE vs LR vs RF vs XGBoost
    # ──────────────────────────────────────────────────────────
    print(f"\n  {'─' * 75}")
    print(f"  Öznitelik Önemi — Rakip Eğitimi")
    print(f"  {'─' * 75}")

    rf_model = fit_competitor_rf(X_train, y_train,
                                 n_estimators=500, seed=SEED)
    print(f"  RF ✓")

    xgb_model = None
    if HAS_XGB:
        xgb_model = XGBClassifier(
            n_estimators=200, max_depth=6,
            learning_rate=0.1, eval_metric='logloss',
            random_state=SEED, verbosity=0)
        xgb_model.fit(X_train, y_train)
        print(f"  XGBoost ✓")
    else:
        print(f"  XGBoost ATLANDI (pip install xgboost)")

    # Tablo A: Ham katsayılar
    raw_coefficients_comparison(flare, lr_model, feature_names)

    # Tablo B: Normalize sıralama + Spearman
    feature_importance_ranking(
        flare, lr_model=lr_model, rf_model=rf_model,
        xgb_model=xgb_model, feature_names=feature_names)

    # ──────────────────────────────────────────────────────────
    # 8. GLOBAL ETKİLEŞİM TESTLERİ
    # ──────────────────────────────────────────────────────────
    interactions = run_interaction_analysis(
        flare, X_train, feature_names,
        report_threshold=0.2, top_n_ref=3)

    # ──────────────────────────────────────────────────────────
    # 9. RİSK GRUPLARI — β_equiv by group
    # ──────────────────────────────────────────────────────────
    beta_equiv_by_group(
        flare, X_test, y_test, feature_names, n_groups=3)

    # ──────────────────────────────────────────────────────────
    # 10. HASTA SEÇİMİ — 4 BELİRSİZLİK KATEGORİSİ + YÜKSEK RİSKLİ CC
    # ──────────────────────────────────────────────────────────

    # CI hesapla
    ci_result = flare.predict_ci(X_test)
    ci_lo = ci_result['ci_lo']
    ci_hi = ci_result['ci_hi']
    se_all = ci_result['se_eta']

    # 4 kategori
    ci_excludes_05 = (ci_lo > 0.5) | (ci_hi < 0.5)
    prediction = (flare_probs >= 0.5).astype(int)
    correct = (prediction == y_test.astype(int))

    cat_cc = np.where(ci_excludes_05 & correct)[0]
    cat_cw = np.where(ci_excludes_05 & ~correct)[0]
    cat_uc = np.where(~ci_excludes_05 & correct)[0]
    cat_uw = np.where(~ci_excludes_05 & ~correct)[0]

    print(f"\n  Kategori havuzları:")
    print(f"    CC: {len(cat_cc)}")
    print(f"    CW: {len(cat_cw)}")
    print(f"    UC: {len(cat_uc)}")
    print(f"    UW: {len(cat_uw)}")

    # CC-Low: düşük riskli, SE yüksek (p̂ < 0.2)
    idx_cc_low = None
    if len(cat_cc) > 0:
        low_cc = cat_cc[flare_probs[cat_cc] < 0.2]
        if len(low_cc) > 0:
            idx_cc_low = low_cc[np.argmax(se_all[low_cc])]

    # CC-High: yüksek riskli, SE yüksek (p̂ > 0.8)
    idx_cc_high = None
    if len(cat_cc) > 0:
        high_cc = cat_cc[flare_probs[cat_cc] > 0.8]
        if len(high_cc) > 0:
            idx_cc_high = high_cc[np.argmax(se_all[high_cc])]

    # CW: p̂ uzak 0.5 ama yanlış, SE düşük (tehlikeli)
    idx_cw = None
    if len(cat_cw) > 0:
        idx_cw = cat_cw[np.argmin(se_all[cat_cw])]

    # UC: p̂ ≈ 0.5'e en yakın
    idx_uc = None
    if len(cat_uc) > 0:
        idx_uc = cat_uc[np.argmin(np.abs(flare_probs[cat_uc] - 0.5))]

    # UW: p̂ ≈ 0.5'e en yakın ama yanlış
    idx_uw = None
    if len(cat_uw) > 0:
        idx_uw = cat_uw[np.argmin(np.abs(flare_probs[cat_uw] - 0.5))]

    selected = [idx for idx in [idx_cc_low, idx_cc_high, idx_cw, idx_uc, idx_uw]
                if idx is not None]

    # Kategori bilgilerini sakla
    patient_categories = {}
    label_map = {
        idx_cc_low: 'CC-Low',
        idx_cc_high: 'CC-High',
        idx_cw: 'CW',
        idx_uc: 'UC',
        idx_uw: 'UW',
    }
    for idx, label in label_map.items():
        if idx is not None:
            patient_categories[idx] = label

    # Yazdır
    print(f"\n  {'─' * 75}")
    print(f"  Seçilen Hastalar (5 Kategori)")
    print(f"  {'─' * 75}")
    print(f"  {'Kategori':<12} {'İndeks':<10} {'p̂':<10} "
          f"{'SE(η)':<10} {'95% CI':<24} {'Doğru':<8} {'Label'}")
    print(f"  {'─' * 75}")
    for idx in selected:
        cat = patient_categories[idx]
        ci_str = f"[{ci_lo[idx]:.4f}, {ci_hi[idx]:.4f}]"
        print(f"  {cat:<12} {idx:<10} {flare_probs[idx]:.4f}  "
              f"{se_all[idx]:.4f}  {ci_str:<24} "
              f"{'✓' if correct[idx] else '✗':<8} "
              f"{int(y_test[idx])}")

    se_vals = [se_all[idx] for idx in selected]
    if len(se_vals) >= 2:
        print(f"\n  SE(η) aralığı  : {min(se_vals):.4f} – "
              f"{max(se_vals):.4f}")
        print(f"  SE(η) oranı    : "
              f"{max(se_vals)/max(min(se_vals), EPS):.2f}×")
    # ──────────────────────────────────────────────────────────
    # 11. HASTA RİSK PROFİLLERİ (4 kategori)
    # ──────────────────────────────────────────────────────────
    for idx in selected:
        label = patient_categories[idx]
        print(f"\n  {'━' * 50}")
        print(f"  Hasta #{idx} — Kategori: {label}")
        print(f"  {'━' * 50}")
        profile = patient_risk_profile(
            flare, X_test, feature_names, idx)
        print_patient_risk_report(profile)

    # ──────────────────────────────────────────────────────────
    # 12. ORF-ATTRIBUTION (4 hasta)
    # ──────────────────────────────────────────────────────────
    for idx in selected:
        rff_attribution_patient(
            flare, X_test, feature_names, idx)

    # ──────────────────────────────────────────────────────────
    # 13. HASTA BAZLI HESSIAN ETKİLEŞİMLERİ
    # ──────────────────────────────────────────────────────────
    hessians_ci, hessians = compute_patient_hessians_with_ci(
        flare, X_test, feature_names, selected, top_n=10)
    print_patient_hessians_with_ci(hessians_ci, flare_probs)

    # ──────────────────────────────────────────────────────────
    # 14. GLOBAL vs HASTA BAZLI KARŞILAŞTIRMA
    # ──────────────────────────────────────────────────────────
    comp_table, H_global = global_vs_patient_interactions(
        flare, X_train, feature_names, selected, hessians)

    # ──────────────────────────────────────────────────────────
    # 15. BENZERSİZ HASTA ETKİLEŞİMLERİ
    # ──────────────────────────────────────────────────────────
    unique_patient_interactions(
        flare, X_train, X_test, feature_names, selected,
        hessians, H_global=H_global, threshold=2.0)

    # ──────────────────────────────────────────────────────────
    # 16. RESİDUAL ANALİZ
    # ──────────────────────────────────────────────────────────
    resid = residuals_summary(flare, X_test, y_test)

    # ──────────────────────────────────────────────────────────
    # 17. POSTERIOR COVERAGE
    # ──────────────────────────────────────────────────────────
    coverage = binary_posterior_coverage(
        flare, X_test, y_test, method='equal_frequency')
    print_coverage_report(coverage)

    # ──────────────────────────────────────────────────────────
    # 18. HASTA OLASI LİK SINIFLANDIRMASI
    # ──────────────────────────────────────────────────────────
    plaus = patient_posterior_plausibility(flare, X_test, y_test)
    print(f"\n  Hasta Olasılık Sınıflandırması:")
    for cat in [1, 2, 3, 4]:
        print(f"    {plaus['labels'][cat]}: "
              f"{plaus['counts'][cat]} "
              f"({plaus['rates'][cat]:.1%})")

    # ──────────────────────────────────────────────────────────
    # 19. TRUE BETA (sentetik veri ise)
    # ──────────────────────────────────────────────────────────
    if 'true_weights' in meta:
        true_beta_comparison(
            meta, flare, lr_model, feature_names)

    # ──────────────────────────────────────────────────────────
    # 20. GÖRSELLEŞTİRME VERİLERİ
    # ──────────────────────────────────────────────────────────
    roc_data = roc_curve_data(flare, X_test, y_test)
    pr_data = pr_curve_data(flare, X_test, y_test)
    cal_data = calibration_plot_data(flare, X_test, y_test)
    resid_data = residuals_plot_data(flare, X_test, y_test)

    print(f"\n  Görselleştirme verileri hazır:")
    print(f"    ROC AUC    = {roc_data['auc']:.4f}")
    print(f"    PR  AP     = {pr_data['ap']:.4f}")
    print(f"    Brier      = {cal_data['brier']:.4f}")
    print(f"    ECE        = {cal_data['ece']:.4f}")

    # ──────────────────────────────────────────────────────────
    # SONUÇ
    # ──────────────────────────────────────────────────────────
    print(f"\n  {'═' * 75}")
    print(f"  ANALİZ TAMAMLANDI")
    print(f"  {'═' * 75}")

    return {
        'flare': flare,
        'lr_model': lr_model,
        'rf_model': rf_model,
        'xgb_model': xgb_model,
        'lr_res': lr_res,
        'beq': beq,
        'interactions': interactions,
        'hessians_ci': hessians_ci,
        'hessians': hessians,
        'H_global': H_global,
        'selected': selected,
        'patient_categories': patient_categories,
        'residuals': resid,
        'coverage': coverage,
        'plausibility': plaus,
        'summary': summary,
        'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': y_test,
        'feature_names': feature_names,
        'meta': meta,
        'roc_data': roc_data,
        'pr_data': pr_data,
        'cal_data': cal_data,
        'resid_data': resid_data,
    }


if __name__ == '__main__':
    result = main()


  ═══════════════════════════════════════════════════════════════════════════
  FLARE — TEK MODEL ANALİZ
  ═══════════════════════════════════════════════════════════════════════════
  Veri       : LINEAR (Synthetic)
  n_total    : 5000
  d          : 8
  pos_rate   : 53.0%
  n_train    : 4000
  n_test     : 1000
  m          : 126
  λ (l2)     : 0.1
  σ          : otomatik
  seed       : 42

  ───────────────────────────────────────────────────────────────────────────
  FLARE Eğitimi
  ───────────────────────────────────────────────────────────────────────────
  σ (seçilen): 2.360076
  AUC        : 0.8364
  PR-AUC     : 0.8529
  F1         : 0.7828
  Accuracy   : 0.7620
  MCC        : 0.5216

  ───────────────────────────────────────────────────────────────────────────
  LR Eğitimi (Wald karşılaştırma için)
  ───────────────────────────────────────────────────────────────────────────
  LR AUC     : 0.8384

  FLARE MODEL SUMMARY
  Samples        = 1000
  Features (d)   = 8
  Classes  